# Notebook 04: Comprehensive Heuristic Function Analysis

## Overview

This notebook provides a deep dive into heuristic functions used in A* search. We'll explore all 5 heuristic functions available in the pathfinding lab:

1. **Manhattan Distance** - L1 norm, perfect for 4-directional movement
2. **Euclidean Distance** - L2 norm, straight-line distance
3. **Chebyshev Distance** - L∞ norm, maximum coordinate difference
4. **Octile Distance** - Optimized for 8-directional movement
5. **Weighted Manhattan** - Configurable weight parameter

## Learning Objectives

By the end of this notebook, you will:
1. Understand the mathematical formula for each heuristic
2. Recognize when each heuristic is admissible
3. Visualize heuristic values on grids
4. Know which heuristic to use for different scenarios
5. Understand the trade-off between heuristic accuracy and computation time

## Key Concepts

### Admissibility
A heuristic h(n) is **admissible** if:
```
h(n) ≤ actual_cost(n, goal) for all n
```
Admissible heuristics never overestimate, guaranteeing A* finds optimal paths.

### Consistency (Monotonicity)
A heuristic is **consistent** if:
```
h(n) ≤ cost(n, n') + h(n') for all neighbors n'
```
Consistent heuristics are always admissible and enable efficient A* implementation.

### Dominance
Heuristic h1 **dominates** h2 if:
```
h1(n) ≥ h2(n) for all n, and both are admissible
```
The dominant heuristic will expand fewer nodes (more informed).

## Setup and Imports

In [ ]:
# Add parent directory to path for imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Core imports
from src.pathfinding_lab.core.grid import Grid
from src.pathfinding_lab.core.types import MovementMode
from src.pathfinding_lab.algorithms.astar import astar
from src.pathfinding_lab.algorithms.dijkstra import dijkstra

# All heuristic functions
from src.pathfinding_lab.heuristics.manhattan import manhattan_distance
from src.pathfinding_lab.heuristics.euclidean import euclidean_distance
from src.pathfinding_lab.heuristics.octile import octile_distance
from src.pathfinding_lab.heuristics.chebyshev import chebyshev_distance
from src.pathfinding_lab.heuristics.weighted import weighted_manhattan_distance

# Visualization
from src.pathfinding_lab.visualization.grid_plot import create_grid_plot
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
print("✓ Imports successful!")

## Part 1: Mathematical Definitions and Formulas

Let's examine each heuristic mathematically.

In [ ]:
# Define test positions
start_pos = (0, 0)
test_positions = [
    (5, 0),   # Vertical
    (0, 5),   # Horizontal
    (3, 4),   # Diagonal
    (5, 5),   # Perfect diagonal
    (10, 7),  # Mixed
]

print("="*80)
print("HEURISTIC FUNCTION FORMULAS AND VALUES")
print("="*80)
print(f"Start position: {start_pos}\n")

for goal_pos in test_positions:
    dx = abs(goal_pos[0] - start_pos[0])
    dy = abs(goal_pos[1] - start_pos[1])
    
    print(f"\nGoal: {goal_pos} (Δx={dx}, Δy={dy})")
    print("-" * 80)
    
    # Manhattan
    manhattan = manhattan_distance(start_pos, goal_pos)
    print(f"Manhattan:  h(n) = |Δx| + |Δy| = {dx} + {dy} = {manhattan:.3f}")
    
    # Euclidean
    euclidean = euclidean_distance(start_pos, goal_pos)
    print(f"Euclidean:  h(n) = √(Δx² + Δy²) = √({dx}² + {dy}²) = {euclidean:.3f}")
    
    # Chebyshev
    chebyshev = chebyshev_distance(start_pos, goal_pos)
    print(f"Chebyshev:  h(n) = max(|Δx|, |Δy|) = max({dx}, {dy}) = {chebyshev:.3f}")
    
    # Octile
    octile = octile_distance(start_pos, goal_pos)
    D1, D2 = 1.0, np.sqrt(2)
    dmin, dmax = min(dx, dy), max(dx, dy)
    print(f"Octile:     h(n) = D₂×min + D₁×(max-min) = {D2:.3f}×{dmin} + {D1:.3f}×{dmax-dmin} = {octile:.3f}")
    
    # Weighted (weight=1.0)
    weighted = weighted_manhattan_distance(start_pos, goal_pos, weight=1.0)
    print(f"Weighted:   h(n) = w × (|Δx| + |Δy|) = 1.0 × {manhattan:.3f} = {weighted:.3f}")

## Part 2: Visualizing Heuristic Values

Let's create heatmaps showing the heuristic values for every cell on a grid.

In [ ]:
# Create heuristic heatmaps
grid_size = 20
goal = (15, 15)

# Compute heuristic values for all cells
heuristics_to_plot = [
    ("Manhattan", manhattan_distance),
    ("Euclidean", euclidean_distance),
    ("Chebyshev", chebyshev_distance),
    ("Octile", octile_distance),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
axes = axes.flatten()

for idx, (name, heuristic_func) in enumerate(heuristics_to_plot):
    # Compute heuristic values
    heatmap = np.zeros((grid_size, grid_size))
    for row in range(grid_size):
        for col in range(grid_size):
            heatmap[row, col] = heuristic_func((row, col), goal)
    
    # Plot heatmap
    im = axes[idx].imshow(heatmap, cmap='YlOrRd', interpolation='nearest')
    axes[idx].set_title(f"{name} Distance to Goal {goal}", fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Column', fontsize=11)
    axes[idx].set_ylabel('Row', fontsize=11)
    
    # Mark goal
    axes[idx].plot(goal[1], goal[0], 'b*', markersize=20, markeredgecolor='white', markeredgewidth=2)
    
    # Add colorbar
    plt.colorbar(im, ax=axes[idx], label='Heuristic Value')
    
    # Add grid
    axes[idx].set_xticks(np.arange(-0.5, grid_size, 1), minor=True)
    axes[idx].set_yticks(np.arange(-0.5, grid_size, 1), minor=True)
    axes[idx].grid(which='minor', color='gray', linestyle='-', linewidth=0.3, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Manhattan: Diamond/rhombus shape (L1 norm)")
print("- Euclidean: Circular contours (L2 norm)")
print("- Chebyshev: Square contours (L∞ norm)")
print("- Octile: Octagonal contours (optimized for 8-directional)")

## Part 3: Admissibility Analysis

Let's verify which heuristics are admissible for different movement modes.

In [ ]:
# Test admissibility by comparing heuristic values to actual shortest path costs
def test_admissibility(movement_mode, heuristic_name, heuristic_func, num_tests=50):
    """
    Test if a heuristic is admissible by comparing h(n) to actual costs.
    """
    overestimates = 0
    max_overestimate = 0.0
    
    for test_num in range(num_tests):
        # Create simple grid (no obstacles for clean test)
        grid = Grid(width=15, height=15, obstacle_density=0.0, 
                    movement_mode=movement_mode, random_seed=test_num)
        
        # Random start and goal
        np.random.seed(test_num)
        start = (np.random.randint(0, 15), np.random.randint(0, 15))
        goal = (np.random.randint(0, 15), np.random.randint(0, 15))
        
        # Get actual optimal cost using Dijkstra
        result = dijkstra(grid, start, goal)
        actual_cost = result.path_cost
        
        # Get heuristic estimate
        h_estimate = heuristic_func(start, goal)
        
        # Check if overestimate
        if h_estimate > actual_cost + 0.001:  # Small epsilon for floating point
            overestimates += 1
            overestimate_amount = h_estimate - actual_cost
            max_overestimate = max(max_overestimate, overestimate_amount)
    
    is_admissible = (overestimates == 0)
    return is_admissible, overestimates, max_overestimate

# Test all heuristics for both movement modes
movement_modes = [
    ("4-Directional", MovementMode.FOUR_DIRECTIONAL),
    ("8-Directional", MovementMode.EIGHT_DIRECTIONAL),
]

heuristics_to_test = [
    ("Manhattan", manhattan_distance),
    ("Euclidean", euclidean_distance),
    ("Chebyshev", chebyshev_distance),
    ("Octile", octile_distance),
]

print("="*80)
print("ADMISSIBILITY TEST RESULTS")
print("="*80)

for mode_name, movement_mode in movement_modes:
    print(f"\n{mode_name} Movement:")
    print("-" * 80)
    print(f"{'Heuristic':<15} {'Admissible?':<15} {'Overestimates':<20} {'Max Overestimate'}")
    print("-" * 80)
    
    for h_name, h_func in heuristics_to_test:
        is_adm, overest, max_overest = test_admissibility(movement_mode, h_name, h_func, num_tests=30)
        adm_str = "✓ Yes" if is_adm else "✗ No"
        print(f"{h_name:<15} {adm_str:<15} {overest}/30{'':<15} {max_overest:.4f}")

print("\n" + "="*80)

## Part 4: Performance Comparison

How do different heuristics perform in practice?

In [ ]:
# Comprehensive performance test
test_grid = Grid(width=30, height=30, obstacle_density=0.2,
                 movement_mode=MovementMode.EIGHT_DIRECTIONAL, random_seed=42)
test_start = (2, 2)
test_goal = (27, 27)
test_grid.generate_obstacles(test_start, test_goal)

# Test all heuristics
print("Testing all heuristics on 30x30 grid with obstacles...\n")
print("="*90)
print(f"{'Heuristic':<20} {'Path Cost':<12} {'Path Length':<12} {'Visited':<10} {'Time (ms)'}")
print("="*90)

results = {}

# Dijkstra baseline (no heuristic)
dijkstra_result = dijkstra(test_grid, test_start, test_goal)
print(f"{'Dijkstra (baseline)':<20} {dijkstra_result.path_cost:<12.3f} {dijkstra_result.path_length:<12} "
      f"{dijkstra_result.nodes_visited:<10} {dijkstra_result.runtime_ms:.2f}")
results['Dijkstra'] = dijkstra_result

print("-" * 90)

# All heuristics
for h_name, h_func in heuristics_to_test:
    result = astar(test_grid, test_start, test_goal, h_func)
    print(f"{h_name:<20} {result.path_cost:<12.3f} {result.path_length:<12} "
          f"{result.nodes_visited:<10} {result.runtime_ms:.2f}")
    results[h_name] = result

# Weighted Manhattan with different weights
print("-" * 90)
for weight in [0.5, 1.0, 1.5, 2.0]:
    def weighted_h(pos1, pos2):
        return weighted_manhattan_distance(pos1, pos2, weight=weight)
    
    result = astar(test_grid, test_start, test_goal, weighted_h)
    name = f"Weighted (w={weight})"
    print(f"{name:<20} {result.path_cost:<12.3f} {result.path_length:<12} "
          f"{result.nodes_visited:<10} {result.runtime_ms:.2f}")
    results[name] = result

print("="*90)

In [ ]:
# Visualize performance metrics
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Extract data for main heuristics only
main_names = ['Dijkstra', 'Manhattan', 'Euclidean', 'Chebyshev', 'Octile']
main_results = {name: results[name] for name in main_names}

names = list(main_results.keys())
visited = [r.nodes_visited for r in main_results.values()]
times = [r.runtime_ms for r in main_results.values()]
costs = [r.path_cost for r in main_results.values()]
lengths = [r.path_length for r in main_results.values()]

colors = ['gray', 'blue', 'green', 'orange', 'red']

# Nodes visited
axes[0, 0].bar(names, visited, color=colors, alpha=0.7, edgecolor='black')
axes[0, 0].set_ylabel('Nodes Visited', fontsize=12)
axes[0, 0].set_title('Search Efficiency (Lower is Better)', fontsize=13, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)

# Runtime
axes[0, 1].bar(names, times, color=colors, alpha=0.7, edgecolor='black')
axes[0, 1].set_ylabel('Runtime (ms)', fontsize=12)
axes[0, 1].set_title('Computation Time (Lower is Better)', fontsize=13, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# Path cost
axes[1, 0].bar(names, costs, color=colors, alpha=0.7, edgecolor='black')
axes[1, 0].set_ylabel('Path Cost', fontsize=12)
axes[1, 0].set_title('Path Optimality (All Should Be Equal)', fontsize=13, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(axis='y', alpha=0.3)

# Efficiency ratio (relative to Dijkstra)
efficiency = [v / visited[0] for v in visited]
axes[1, 1].bar(names, efficiency, color=colors, alpha=0.7, edgecolor='black')
axes[1, 1].axhline(y=1.0, color='red', linestyle='--', label='Dijkstra baseline')
axes[1, 1].set_ylabel('Relative Efficiency', fontsize=12)
axes[1, 1].set_title('Efficiency vs Dijkstra (Lower is Better)', fontsize=13, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print efficiency improvements
print("\nEfficiency Improvements over Dijkstra:")
for name, eff in zip(names[1:], efficiency[1:]):
    improvement = (1 - eff) * 100
    print(f"{name:15s}: {improvement:5.1f}% fewer nodes visited")

## Part 5: Weighted Heuristics and Speed vs Optimality Trade-off

Weighted heuristics sacrifice optimality for speed.

In [ ]:
# Test weighted heuristics with various weights
weights_to_test = np.arange(0.5, 3.1, 0.25)

weighted_costs = []
weighted_visited = []
weighted_times = []

optimal_cost = dijkstra_result.path_cost

print(f"Optimal path cost (Dijkstra): {optimal_cost:.3f}\n")
print("Testing weighted Manhattan heuristic with different weights...\n")
print(f"{'Weight':<10} {'Path Cost':<15} {'Suboptimality %':<18} {'Visited':<10} {'Time (ms)'}")
print("-" * 70)

for w in weights_to_test:
    def weighted_h(pos1, pos2):
        return weighted_manhattan_distance(pos1, pos2, weight=w)
    
    result = astar(test_grid, test_start, test_goal, weighted_h)
    
    suboptimality = ((result.path_cost - optimal_cost) / optimal_cost) * 100
    
    weighted_costs.append(result.path_cost)
    weighted_visited.append(result.nodes_visited)
    weighted_times.append(result.runtime_ms)
    
    print(f"{w:<10.2f} {result.path_cost:<15.3f} {suboptimality:<18.2f} {result.nodes_visited:<10} {result.runtime_ms:.2f}")

In [ ]:
# Plot speed vs optimality trade-off
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Nodes visited vs weight
axes[0].plot(weights_to_test, weighted_visited, marker='o', linewidth=2.5, markersize=6)
axes[0].axhline(y=dijkstra_result.nodes_visited, color='red', linestyle='--', 
                label='Dijkstra baseline', linewidth=2)
axes[0].set_xlabel('Weight', fontsize=12)
axes[0].set_ylabel('Nodes Visited', fontsize=12)
axes[0].set_title('Search Efficiency vs Weight', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Path cost vs weight
suboptimality_pct = [((c - optimal_cost) / optimal_cost) * 100 for c in weighted_costs]
axes[1].plot(weights_to_test, suboptimality_pct, marker='s', linewidth=2.5, 
             markersize=6, color='orange')
axes[1].axhline(y=0, color='green', linestyle='--', label='Optimal', linewidth=2)
axes[1].set_xlabel('Weight', fontsize=12)
axes[1].set_ylabel('Suboptimality (%)', fontsize=12)
axes[1].set_title('Path Quality vs Weight', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Trade-off: nodes visited vs suboptimality
axes[2].scatter(suboptimality_pct, weighted_visited, c=weights_to_test, 
                cmap='viridis', s=100, edgecolors='black', linewidths=1.5)
axes[2].set_xlabel('Suboptimality (%)', fontsize=12)
axes[2].set_ylabel('Nodes Visited', fontsize=12)
axes[2].set_title('Speed vs Optimality Trade-off', fontsize=13, fontweight='bold')
cbar = plt.colorbar(axes[2].collections[0], ax=axes[2])
cbar.set_label('Weight', fontsize=11)
axes[2].grid(True, alpha=0.3)

# Annotate sweet spots
axes[2].annotate('Optimal (w≤1.0)', xy=(0, weighted_visited[2]), 
                xytext=(5, weighted_visited[2] + 50),
                arrowprops=dict(arrowstyle='->', color='green', lw=2),
                fontsize=10, fontweight='bold', color='green')

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("Weights > 1.0 make the heuristic inadmissible, trading optimality for speed.")
print("This is called 'Weighted A*' and is useful when speed matters more than optimality.")

## Part 6: Choosing the Right Heuristic

Decision guide for selecting heuristics.

In [ ]:
# Test all heuristics on different movement modes
scenarios = [
    ("4-Directional (Cardinal only)", MovementMode.FOUR_DIRECTIONAL),
    ("8-Directional (With diagonals)", MovementMode.EIGHT_DIRECTIONAL),
]

print("="*100)
print("HEURISTIC SELECTION GUIDE")
print("="*100)

for scenario_name, movement_mode in scenarios:
    print(f"\n{scenario_name}")
    print("-" * 100)
    
    # Create test grid
    scenario_grid = Grid(width=25, height=25, obstacle_density=0.15,
                         movement_mode=movement_mode, random_seed=42)
    scenario_start = (2, 2)
    scenario_goal = (22, 22)
    scenario_grid.generate_obstacles(scenario_start, scenario_goal)
    
    # Get optimal cost
    opt_result = dijkstra(scenario_grid, scenario_start, scenario_goal)
    
    print(f"{'Heuristic':<15} {'Admissible?':<12} {'Visited':<10} {'Efficiency':<12} {'Recommendation'}")
    print("-" * 100)
    
    best_visited = float('inf')
    best_heuristic = None
    
    for h_name, h_func in heuristics_to_test:
        result = astar(scenario_grid, scenario_start, scenario_goal, h_func)
        is_optimal = abs(result.path_cost - opt_result.path_cost) < 0.01
        admissible_str = "✓ Yes" if is_optimal else "✗ No"
        efficiency = result.nodes_visited / opt_result.nodes_visited
        
        if is_optimal and result.nodes_visited < best_visited:
            best_visited = result.nodes_visited
            best_heuristic = h_name
        
        recommendation = ""
        if h_name == best_heuristic:
            recommendation = "★ BEST CHOICE ★"
        elif not is_optimal:
            recommendation = "❌ Not admissible"
        
        print(f"{h_name:<15} {admissible_str:<12} {result.nodes_visited:<10} {efficiency:<12.3f} {recommendation}")

print("\n" + "="*100)

## Exercise 1: Custom Heuristic

Create your own heuristic function and test its admissibility.

In [ ]:
# Your code here
# Create a custom heuristic function
# Test if it's admissible
# Compare its performance to existing heuristics

def custom_heuristic(pos1, pos2):
    """
    Your custom heuristic here.
    Try to make it admissible and efficient!
    """
    # Example: average of Manhattan and Euclidean
    manhattan = manhattan_distance(pos1, pos2)
    euclidean = euclidean_distance(pos1, pos2)
    return (manhattan + euclidean) / 2.0

# Test your heuristic
# ...

## Exercise 2: Heuristic Visualization

Create a visualization showing which cells A* explores with different heuristics.

In [ ]:
# Your code here
# Run A* with different heuristics
# Create a side-by-side visualization showing visited cells
# Color cells by the order they were visited

# ...

## Key Takeaways

### Heuristic Selection Guide:

| Movement Mode | Best Heuristic | Reason |
|---------------|----------------|--------|
| 4-Directional | **Manhattan** | Exact distance for cardinal-only movement |
| 8-Directional | **Octile** | Tight lower bound accounting for diagonals |
| Any | **Euclidean** | Safe, admissible straight-line distance |
| Speed > Optimality | **Weighted (w>1)** | Faster but may find suboptimal paths |

### Admissibility Summary:
- **Always Admissible**: Manhattan, Euclidean, Octile (when used correctly)
- **Sometimes Admissible**: Chebyshev (only for 8-directional with equal cardinal/diagonal costs)
- **Configurable**: Weighted Manhattan (admissible if weight ≤ 1.0)

### Performance Insights:
1. **Better heuristic = fewer nodes explored**
2. **Tighter lower bound = more informed search**
3. **Inadmissible heuristics trade optimality for speed**
4. **Computation cost matters**: Simple heuristics (Manhattan) can be faster than complex ones

### When Heuristics Don't Help:
- Highly constrained environments (many obstacles)
- No clear path to goal
- Very short distances (overhead dominates)

## Next Steps

In the next notebook (05_algorithm_comparison.ipynb), we'll:
- Benchmark ALL algorithms on the same grids
- Create comprehensive comparison tables
- Generate performance plots
- Provide recommendations for different scenarios

## Further Reading

- [Heuristics in Pathfinding](http://theory.stanford.edu/~amitp/GameProgramming/Heuristics.html)
- [Distance Metrics](https://en.wikipedia.org/wiki/Distance#Mathematical_formalization)
- Russell & Norvig, "Artificial Intelligence: A Modern Approach", Chapter 3